# Task 1.2 — Key Assumptions


## Assumption 1: Symmetry is a Required Property of the Task

### Statement
The method assumes that the pairwise decision function must be **symmetric** — i.e., classifying the pair $(a, b)$ should give the same result as classifying $(b, a)$. In other words, the order in which two inputs are presented to the model should not change the prediction.

### Why the Method Needs It
The entire contribution of the paper — balanced pairwise kernels ($K_{DL}$, $K_{TL}$, $K_{ML}$, $K_{TM}$) — is built to enforce this symmetry property. If symmetry were not required, there would be no need for the linearisation trick that forms the paper's core idea. The balanced kernels are more expensive to compute than plain pairwise kernels, so this design choice is only justified when symmetry is genuinely needed.

### Violation Scenario
Consider a **plagiarism detection** task: given two documents $A$ and $B$, determine whether $B$ copies from $A$. This is inherently asymmetric — $A$ is the original and $B$ is the suspect copy. Applying the balanced kernel here would destroy this directional information and hurt performance, because the symmetry assumption does not hold.

### Paper Citation
Section 1, paragraph 2: *"...the decision function should be symmetric, i.e., $f(a,b) = f(b,a)$ for all $(a,b)$."* Also Theorem 2 (Section 3), which formally proves that balanced kernels achieve this symmetry.

## Assumption 2: The Base Kernel is a Valid Positive Semi-Definite (PSD) Kernel

### Statement
The method assumes that the underlying base kernel $k(\cdot, \cdot)$ used to construct the pairwise kernel is a **valid, positive semi-definite (Mercer) kernel**. Examples in the paper include RBF, polynomial, and additive chi-squared kernels.

### Why the Method Needs It
The entire SVM framework requires the kernel matrix to be PSD to guarantee a unique convex quadratic programming solution. The pairwise kernels $K_D$, $K_{DL}$, etc. are built from $k$ using sums and products. These operations preserve PSD-ness only if $k$ itself is PSD. The paper explicitly proves that the proposed balanced kernels are PSD kernels (Lemma 1, Section 2), relying on this base assumption.

### Violation Scenario
If one used a **non-PSD similarity measure** as the base kernel — for example, a raw edit distance on strings without any transformation — the resulting pairwise kernel matrix may not be PSD. This would cause the SVM solver to fail or produce an incorrect solution, since the convexity guarantee breaks down.

### Paper Citation
Lemma 1 (Section 2): *"If $k$ is a valid kernel, then $K_D$ and $K_T$ are valid kernels."* — The entire derivation assumes $k$ satisfies Mercer's condition.

## Assumption 3: The Training Pair Distribution is Balanced (or Can Be Made Balanced)

### Statement
The method assumes that the set of training pairs can be constructed such that **same-class and different-class pairs are reasonably balanced** in number. A fully symmetric training set (where every pair $(a,b)$ is accompanied by $(b,a)$) is used in Section 3 to prove equivalence with balanced kernels.

### Why the Method Needs It
Theorem 3 of the paper proves that using a **symmetric training set** with a plain pairwise kernel produces the same decision function as using a balanced kernel with a non-symmetric training set. This interchangeability result — which is central to the paper — only holds when the training set satisfies strict symmetry. Moreover, severe class imbalance in pairs (e.g., vastly more negative pairs than positive ones) can bias the SVM, a problem the paper does not directly address.

### Violation Scenario
In a **rare event detection** scenario (e.g., identifying if two medical scans belong to the same rare disease subtype), positive pairs (same rare subtype) are far fewer than negative pairs. The training set cannot be made symmetric in a useful way, and the equivalence proof in Theorem 3 breaks down, leading to a biased model.

### Paper Citation
Section 3, Theorem 3: *"Training with a symmetric training set $\hat{S}$ using $K_D$ yields the same decision function as training with $S$ using $K_{DL}$."* — and the discussion following it, which notes the constraint of requiring a paired dataset.

## Assumption 4 (Bonus): The Feature Representation is Sufficiently Discriminative

### Statement
The paper implicitly assumes that the **input feature representations** (e.g., SIFT, LBP, TPLBP for faces) carry enough discriminative information about class identity, so that measuring similarity in feature space is meaningful for the pairwise classification task.

### Why the Method Needs It
The pairwise kernel $k(a,c)$ computes similarity between feature vectors. If the features are not informative (e.g., raw pixel values without alignment in a face dataset), the kernel similarity will not reflect true identity relationships, and the SVM will learn nothing useful regardless of how sophisticated the pairwise kernel is.

### Violation Scenario
If the input images are **unaligned or have large pose/lighting variation** without normalisation, SIFT descriptors (which are somewhat invariant) may still encode meaningful structure, but raw pixel features would fail completely. The paper would not generalise in this case.

### Paper Citation
Section 5 (Experiments): The paper explicitly evaluates SIFT, LBP, and TPLBP features and shows that the kernel method's quality depends heavily on feature quality — TPLBP features outperform the others in most conditions.

The code above illustrates **Assumption 2**: a proper RBF kernel produces a PSD matrix (all non-negative eigenvalues), while an arbitrary similarity matrix can have negative eigenvalues — violating the requirement for SVM to have a valid solution.